In [ ]:
"""
Images
The introduction of convolutional neural networks revolutionized computer vision8,
and image-based systems have since acquired a new set of capabilities. Problems that
required complex pipelines of highly tuned algorithmic building blocks became solvable
at unprecedented levels of performance by training end-to-end networks with
paired input-and-desired-output examples. To participate in this revolution, you need
to be able to load images from common image formats and then transform the data
into a tensor representation that has the various parts of the image arranged in the
way that PyTorch expects.

An image is represented as a collection of scalars arranged in a regular grid, having
a height and a width (in pixels). You might have a single scalar per grid point (the
pixel), which would be represented as a grayscale image, or multiple scalars per grid
point, which typically represent different colors or different features, such as depth
from a depth camera.

Scalars representing values at individual pixels are often encoded with 8-bit integers,
as in consumer cameras, for example. In medical, scientific, and industrial applications,
you not infrequently find pixels with higher numerical precision, such as 12-
bit and 16-bit. This precision provides a wider range or increased sensitivity in cases in
which the pixel encodes information on a physical property, such as bone density,
temperature, or depth.

You have several ways of encoding numbers into colors.9* The most common is
RGB, which defines a color with three numbers that represent the intensity of red,
green and blue. You can think of a color channel as being a grayscale intensity map of
only the color in question, similar to what you’d see if you looked at the scene in question
through a pair of pure-red sunglasses. Figure 3.3 shows a rainbow in which each
of the RGB channels captures a certain portion of the spectrum. (The figure is simplified,
in that it elides things. The orange and yellow bands, for example, are represented
as a combination of red and green.)

Images come in several file formats, but luckily, you have plenty of ways to load
images in Python. Start by loading a PNG image with the imageio module. You’ll use
imageio throughout the chapter because it handles different data types with a uniform
API. Now load an image, as in the following listing.

8 https://en.wikipedia.org/wiki/Convolutional_neural_network#History
9 Something of an understatement: https://en.wikipedia.org/wiki/Color_model
"""

In [1]:
import torch
# import imageio   # depricated
import imageio.v2 as imageio  # or call imageio.v2.imread
import os
from os import path


img_arr = imageio.imread('bobby.jpg')
img_arr.shape

(720, 1280, 3)

In [ ]:
"""
At this point, img is a NumPy array-like object with three dimensions: two spatial
dimensions (width and height) and a third dimension corresponding to the channels
red, green, and blue. Any library that outputs a NumPy array does so to obtain a
PyTorch tensor. The only thing to watch out for is the layout of dimensions. PyTorch
modules that deal with image data require tensors to be laid out as C x H x W (channels,
height, and width, respectively).

You can use the transpose function to get to an appropriate layout. Given an input
tensor W x H x C, you get to a proper layout by swapping the first and last channels:
"""

In [2]:
img = torch.from_numpy(img_arr)
out = torch.transpose(img, 0, 2)
out.shape

torch.Size([3, 1280, 720])

In [ ]:
"""
You’ve seen this example before, but note that this operation doesn’t make a copy of the
tensor data. Instead, out uses the same underlying storage as img and plays with the size
and stride information at the tensor level. This arrangement is convenient because the
operation is cheap, but (heads up) changing a pixel in img leads to a change in out.

Also note that other deep learning frameworks use different layouts. Originally,
TensorFlow kept the channel dimension last, resulting in a H x W x C layout. (Now it
supports multiple layouts.) This strategy has pros and cons from a low-level performance
standpoint, but it doesn’t make a difference to you as long as you reshape your
tensors properly.

So far, you’ve described a single image. Following the same strategy that you used
for earlier data types, to create a data set of multiple images to use as an input for your
neural networks, you store the images in a batch along the first dimension to obtain a
N x C x H x W tensor.
As a more efficient alternative to using stack to build up the tensor, you can preallocate
a tensor of appropriate size and fill it with images loaded from a directory,
"""

In [3]:
batch_size = 100
batch = torch.zeros(100, 3, 256, 256, dtype=torch.uint8)
batch.shape
# batch

torch.Size([100, 3, 256, 256])

In [ ]:
"""
which indicates that your batch will consist of 100 RGB images 256 pixels in height
and 256 pixels in width. Notice the type of the tensor: you’re expecting each color to
be represented as a 8-bit integer, as in most photographic formats from standard consumer
cameras. Now you can load all png images from an input directory and store
them in the tensor:

https://github.com/deep-learning-with-pytorch/dlwpt-code/find/master
"""

In [4]:
# from torchdata.datapipes.iter import FileLister
# from torchvision import io

data_dir = 'C:/Users/jay_s/PyTorch/Tensor/3_Real_world_data/cat_images'  
# dp = Filelister(data_dir, recursive=False)

filenames = [name for name in os.listdir(data_dir) if os.path.splitext(name)[-1] == '.png']
print(filenames)

# cfilenames = [name for name in os.listdir(data_dir) if path.splitext(name)[-1] == '.csv']
# print(cfilenames)


for i, filename in enumerate(filenames):
    img_arr = imageio.imread(filename)
    batch[i] = torch.transpose(torch.from_numpy(img_arr), 0, 2)



['cat1.png', 'cat2.png', 'cat3.png']


In [5]:
# Another way to do the above code
import os
from os import path

"""
data_dir = os.getcwd()
data_dir = os.chdir('..\cat_images')
filenames= []
for name in os.listdir(data_dir):
    txt = os.path.splitext(name)
    if txt[1] == '.png':
        filenames += [name]

# print(filenames)       # ['cat1.png', 'cat2.png', 'cat3.png']

"""


# But I need filenmae in the format ('cat1', '.png') , so code changed to
data_dir = os.getcwd()
# print('data_dir : ', data_dir)

# data_dir = os.chdir('../cat_images')
data_dir = os.chdir('cat_images')
filenames= []

for name in os.listdir(data_dir):
    txt = os.path.splitext(name)
    if txt[1] == '.png':
        filenames += [name]
print(filenames)

for i, filename in enumerate(filenames):
    img_arr = imageio.imread(filename)
    batch[i] = torch.transpose(torch.from_numpy(img_arr), 0, 2)



['cat1.png', 'cat2.png', 'cat3.png']


In [ ]:
"""
As we mentioned earlier, neural networks usually work with floating-point tensors as
their input. As you’ll also see in upcoming chapters, neural networks exhibit the best
training performance when input data ranges from roughly 0 to 1 or –1 to 1 (an effect
of how their building blocks are defined).
A typical thing that you’ll want to do is cast a tensor to floating-point and normalize
the values of the pixels. Casting to floating-point is easy, but normalization is trickier,
as it depends on what range of the input you decide should lie between 0 and 1
(or –1 and 1). One possibility is to divide the values of pixels by 255 (the maximum
representable number in 8-bit unsigned):

"""

In [6]:
batch = batch.float()
batch /= 255.0
batch

tensor([[[[0.6118, 0.6824, 0.4980,  ..., 0.4549, 0.5059, 0.5059],
          [0.5961, 0.5255, 0.6118,  ..., 0.5098, 0.5098, 0.4824],
          [0.4863, 0.6471, 0.4196,  ..., 0.5059, 0.4824, 0.4627],
          ...,
          [0.5882, 0.4706, 0.5137,  ..., 0.4980, 0.4510, 0.4431],
          [0.5843, 0.5333, 0.5608,  ..., 0.4627, 0.4745, 0.4745],
          [0.6196, 0.5412, 0.6431,  ..., 0.4392, 0.4471, 0.4706]],

         [[0.5451, 0.6275, 0.4431,  ..., 0.3882, 0.4353, 0.4353],
          [0.5294, 0.4667, 0.5490,  ..., 0.4314, 0.4353, 0.4078],
          [0.4275, 0.5843, 0.3529,  ..., 0.4353, 0.4157, 0.4000],
          ...,
          [0.5294, 0.4118, 0.4627,  ..., 0.4588, 0.4157, 0.4039],
          [0.5294, 0.4784, 0.5059,  ..., 0.4235, 0.4392, 0.4314],
          [0.5765, 0.4863, 0.5961,  ..., 0.4039, 0.4118, 0.4353]],

         [[0.5059, 0.6078, 0.4078,  ..., 0.3647, 0.4235, 0.4196],
          [0.4824, 0.4314, 0.5176,  ..., 0.4235, 0.4235, 0.3843],
          [0.3843, 0.5373, 0.3137,  ..., 0

In [ ]:
"""
Another possibility is to compute mean and standard deviation of the input data and
scale it so that the output has zero mean and unit standard deviation across each
channel:
"""

In [7]:
n_channels = batch.shape[1]
# print(batch.shape[1])            #  3
for c in range(n_channels):
    mean = torch.mean(batch[:, c])
    std = torch.std(batch[:, c])
    batch[:, c] = (batch[:, c] - mean) / std

print(batch)

tensor([[[[ 5.6026,  6.2680,  4.5306,  ...,  4.1240,  4.6045,  4.6045],
          [ 5.4547,  4.7894,  5.6026,  ...,  4.6415,  4.6415,  4.3827],
          [ 4.4197,  5.9353,  3.7913,  ...,  4.6045,  4.3827,  4.1979],
          ...,
          [ 5.3808,  4.2718,  4.6785,  ...,  4.5306,  4.0870,  4.0131],
          [ 5.3438,  4.8633,  5.1221,  ...,  4.1979,  4.3088,  4.3088],
          [ 5.6765,  4.9372,  5.8983,  ...,  3.9761,  4.0501,  4.2718]],

         [[ 6.2838,  7.2572,  5.0786,  ...,  4.4296,  4.9859,  4.9859],
          [ 6.0983,  5.3567,  6.3301,  ...,  4.9395,  4.9859,  4.6614],
          [ 4.8932,  6.7473,  4.0125,  ...,  4.9859,  4.7541,  4.5687],
          ...,
          [ 6.0983,  4.7078,  5.3103,  ...,  5.2640,  4.7541,  4.6150],
          [ 6.0983,  5.4958,  5.8202,  ...,  4.8468,  5.0322,  4.9395],
          [ 6.6546,  5.5885,  6.8864,  ...,  4.6150,  4.7078,  4.9859]],

         [[ 7.1517,  8.6242,  5.7359,  ...,  5.1129,  5.9624,  5.9058],
          [ 6.8119,  6.0757,  

In [ ]:
"""
You can perform several other operations on inputs, including geometric transformations
such as rotation, scaling, and cropping. These operations may help with
training or may be required to make an arbitrary input conform to the input
requirements of a network, such as the size of the image. You’ll stumble onto quite a
few of these strategies. For now, just remember that you have image manipulation
options available.
"""

In [ ]:
"""
3.5 Volumetric data
You’ve learned how to load and represent 2D images, like the ones you take with your
camera. In contexts such as medical imaging applications involving, say, CT (Computed
Tomography) scans, you typically deal with sequences of images stacked along
the head-to-feet direction, each corresponding to a slice across the body. In CT scans,
the intensity represents the density of the different parts of the body: lungs, fat, water,
muscle, bone, in order of increasing density, mapped from dark to bright when CT
scans are displayed on clinical workstations. The density at each point is computed
from the amount of x-ray reaching a detector after passing through the body, with
some complex math used to deconvolve the raw sensor data into the full volume.
CTs have a single intensity channel, similar to a grayscale image. Often, in native
data formats, the channel dimension is left out, so the raw data typically has three
dimensions. By stacking individual 2D slices into a 3D tensor, you can build volumetric
data representing the 3D anatomy of a subject. Unlike figure 3.3, the extra dimension
in figure 3.4 represents an offset in physical space rather than a particular band of the
visible spectrum.

Figure 3.4 Slices of a CT scan, from the top of the head to the jawline

We won’t go into detail here on medical imaging data formats. For now, it suffices
to say that no fundamental difference exists between a tensor that stores volumetric
data and one that stores image data. You have an extra dimension, depth, after the
channel dimension, leading to a 5D tensor of shape N x C x D x H x W.

Load a sample CT scan by using the volread function in the imageio module, which
takes a directory as argument and assembles all DICOM (Digital Imaging Communication
and Storage) files10 in a series in a NumPy 3D array, as shown in the following listing.


Listing 3.5 code/p1ch4/6_volumetric_ct.ipynb

data/p1ch4/volumetric-dicom/2-LUNG 3.0  B70f-04083

"""

In [8]:
import imageio.v2 as imageio  # or call imageio.v2.imread

# dir_path = "../data/plch4/volumetric-dicom/2-LUNG 3.0 B70f-04083"
# https://github.com/deep-learning-with-pytorch/dlwpt-code/tree/master/data/p1ch4/volumetric-dicom/2-LUNG%203.0%20%20B70f-04083

# dir_path = "https://github.com/deep-learning-with-pytorch/dlwpt-code/tree/master/data/p1ch4/volumetric-dicom/2-LUNG%203.0%20%20B70f-04083"
# dir_path = `https://github.com/deep-learning-with-pytorch/dlwpt-code/tree/master/data/p1ch4/volumetric-dicom/2-LUNG%203.0%20%20B70f-04083`
# dir_path = 'https://github.com/deep-learning-with-pytorch/dlwpt-code/tree/master/data/p1ch4/volumetric-dicom/2-LUNG%203.0%20%20B70f-04083'

dir_path = 'C:/Users/jay_s/PyTorch/Tensor/3_Real_world_data/DICOM' 
vol_arr = imageio.volread(dir_path, 'DICOM')
vol_arr.shape


Reading DICOM (examining files): 1/99 files (1.0%99/99 files (100.0%)
  Found 1 correct series.
Reading DICOM (loading data): 99/99  (100.0%)


(99, 512, 512)

In [ ]:
"""
Also in this case, the layout is different from what PyTorch expects, due to the lack of
channel information. You’ll have to make room for the channel dimension by using
unsqueeze:

"""

In [9]:
vol = torch.from_numpy(vol_arr).float()
vol = torch.transpose(vol, 0, 2)
vol = torch.unsqueeze(vol, 0)
vol.shape

torch.Size([1, 512, 512, 99])

In [ ]:
"""
At this point, you could assemble a 5D data set by stacking multiple volumes along the
batch direction, as you did earlier in the chapter.

Conclusion
You covered a lot of ground in this chapter. You learned to load the most common
types of data and shape them up for consumption by a neural network. There are
more data formats in the wild than we could hope to describe in a single volume, of
course. Some, like medical histories, are too complex to cover in this volume. For the
interested reader, however, we do provide short examples of audio and video tensor
creation in bonus Jupyter notebooks in our code repository11.


Summary
􀂃 Neural networks require data to be represented as multidimensional numerical
tensors, often 32-bit floating-point.
􀂃 Thanks to how the PyTorch libraries interact with the Python standard library
and surrounding ecosystem, loading the most common types of data and converting
them to PyTorch tensors is convenient.
􀂃 In general, PyTorch expects data to be laid out along specific dimensions,
according to the model architecture (such as convolutional versus recurrent).
Data reshaping can be achieved effectively with the PyTorch tensor API.
􀂃 Spreadsheets can be straightforward to convert to tensors. Categorical- and ordinal-
valued columns should be handled differently from interval-valued columns.
􀂃 Text or categorical data can be encoded to a one-hot representation through
the use of dictionaries.
􀂃 Images can have one or many channels. The most common are the red, green,
and blue channels of typical digital photos.
􀂃 Single-channel data formats sometimes omit an explicit channel dimension.
􀂃 Volumetric data is similar to 2D image data, with the exception of adding a
third dimension: depth.
􀂃 Many images have a per-channel bit depth of 8, though 12 and 16 bits per channel
are not uncommon. These bit-depths can be stored in a 32-bit floating-point
number without loss of precision.

"""